# I. THƯ VIỆN &amp; CẤU HÌNH MÔI TRƯỜNG

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thư viện cho Feature Selection và Mô hình hóa
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Cấu hình môi trường hiển thị
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 100)

print("Đã nạp thành công các thư viện phân tích và mô hình hóa!")

Đã nạp thành công các thư viện phân tích và mô hình hóa!


# II. TIỀN XỬ LÝ DỮ LIỆU (Kế thừa Lab 03)
> - Mục đích: Thực thi lại quy trình làm sạch, điền khuyết, loại bỏ ngoại lệ và Feature Engineering đã được chốt ở Lab 03.
> - Bổ sung: Áp dụng One-Hot Encoding cho các biến định tính (Categorical) để thuật toán Machine Learning có thể đọc được.

In [ ]:
# 1. Nạp dữ liệu
csv_files = glob.glob('/kaggle/input/**/train.csv', recursive=True) + glob.glob('/kaggle/input/**/*.csv', recursive=True)
data_path = csv_files[0] if csv_files else "https://raw.githubusercontent.com/selva86/datasets/master/AmesHousing.csv"
df = pd.read_csv(data_path)
df.columns = [col.replace(' ', '') for col in df.columns]

# 2. Ép kiểu dữ liệu rời rạc
retype_cols = ['MSSubClass', 'MoSold', 'YrSold']
for col in retype_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)

# 3. Điền khuyết dữ liệu
cat_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
df[cat_none] = df[cat_none].fillna('None')

num_zero = ['GarageArea', 'GarageCars', 'GarageYrBlt', 'TotalBsmtSF', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in num_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

if 'LotFrontage' in df.columns and 'Neighborhood' in df.columns:
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Điền Mode cho các biến còn lại
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# 4. Feature Engineering & Biến đổi Logarit
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
df['HouseAge'] = df['YrSold'].astype(int) - df['YearBuilt']
df['HouseAge'] = df['HouseAge'].apply(lambda x: max(0, x))
df['RemodAge'] = df['YrSold'].astype(int) - df['YearRemodAdd']
df['RemodAge'] = df['RemodAge'].apply(lambda x: max(0, x))
df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)

df['LogSalePrice'] = np.log1p(df['SalePrice'])

# 5. Loại bỏ điểm đòn bẩy
df = df[~df['Id'].isin([524, 1299])].reset_index(drop=True)

# 6. Mã hóa thứ tự (Ordinal Encoding)
qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
for col in ordinal_cols:
    df[f'{col}_Num'] = df[col].map(qual_mapping).fillna(0)

# Loại bỏ các cột trùng lặp thông tin hoặc đã mã hóa
cols_to_drop = ['Id', 'SalePrice', 'GarageArea', '1stFlrSF'] + ordinal_cols
df_cleaned = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Mã hóa One-Hot (One-Hot Encoding) cho các biến định tính còn lại
df_encoded = pd.get_dummies(df_cleaned, drop_first=True)

print(f"Kích thước dữ liệu sau tiền xử lý: {df_encoded.shape}")

Kích thước dữ liệu sau tiền xử lý: (1458, 260)


# III. FEATURE SELECTION VÀ CHUẨN BỊ DỮ LIỆU
> - Mục đích: Lọc ra các đặc trưng mang tính dự báo cao nhất nhằm giảm thiểu nhiễu và tránh overfitting.
> - Kỹ thuật áp dụng: Sử dụng `SelectFromModel` kết hợp với `RandomForestRegressor` để đánh giá tầm quan trọng của các đặc trưng (Feature Importance).

In [4]:
# Tách biến độc lập (X) và biến mục tiêu (y)
X = df_encoded.drop('LogSalePrice', axis=1)
y = df_encoded['LogSalePrice']

# Phân chia tập Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Khởi tạo mô hình RandomForest để đánh giá Feature Importance
selector_rf = RandomForestRegressor(n_estimators=100, random_state=42)
selector_rf.fit(X_train, y_train)

# Áp dụng SelectFromModel (chọn các biến có độ quan trọng lớn hơn mức trung bình)
selector = SelectFromModel(selector_rf, prefit=True)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

selected_features = X.columns[(selector.get_support())]
print(f"Số lượng đặc trưng ban đầu: {X_train.shape[1]}")
print(f"Số lượng đặc trưng được chọn: {X_train_selected.shape[1]}")
print(f"Top 10 đặc trưng quan trọng nhất: {list(selected_features[:10])}")

Số lượng đặc trưng ban đầu: 259
Số lượng đặc trưng được chọn: 21
Top 10 đặc trưng quan trọng nhất: ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF', 'GrLivArea']


# IV. ĐÁNH GIÁ HIỆU SUẤT MÔ HÌNH (SVM, XGBoost, RandomForest)
> - Mục đích: Huấn luyện các mô hình cơ bản trên tập dữ liệu đã được tinh gọn (Feature Selected).
> - Đánh giá: Sử dụng thước đo RMSE (Root Mean Squared Error) và R2 Score trên thang đo Logarit. Lưu ý, SVM nhạy cảm với thang đo nên cần Standard Scaler.

In [5]:
# Chuẩn hóa dữ liệu cho SVM (Support Vector Machine)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

# Khởi tạo danh sách các mô hình
models = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    "SVM": SVR(kernel='rbf', C=10, gamma='scale')
}

# Đánh giá hiệu suất
results = []
for name, model in models.items():
    # SVM dùng dữ liệu đã Scale, các mô hình Tree-based dùng dữ liệu bình thường
    train_x = X_train_scaled if name == "SVM" else X_train_selected
    test_x = X_test_scaled if name == "SVM" else X_test_selected
    
    # Huấn luyện
    model.fit(train_x, y_train)
    
    # Dự đoán
    y_pred = model.predict(test_x)
    
    # Tính toán Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "RMSE (Log Scale)": rmse,
        "R2 Score": r2
    })

# Hiển thị kết quả dạng bảng
df_results = pd.DataFrame(results).sort_values(by="RMSE (Log Scale)")
display(df_results)

,Model,RMSE (Log Scale),R2 Score
2,SVM,0.141019,0.882034
1,XGBoost,0.141200,0.881731
0,RandomForest,0.146741,0.872266


# TỔNG KẾT
1. **Tiền xử lý:** Kế thừa thành công logic từ Lab 03, biến đổi các đặc trưng không gian (TotalSF, LotFrontage)[cite: 2] và loại trừ hiện tượng đa cộng tuyến.
2. **Feature Selection:** Mô hình RandomForest đã giảm số lượng chiều dữ liệu đáng kể, tập trung vào các biến lõi như TotalSF, OverallQual.
3. **Hiệu suất Mô hình:** Các thuật toán Tree-based (đặc biệt là XGBoost) thường cho độ sai số (RMSE) thấp nhất và R2 cao nhất nhờ khả năng nắm bắt quan hệ phi tuyến tính mạnh mẽ từ tập đặc trưng đã chọn lọc.

# V. NHẬN XÉT VÀ ĐÁNH GIÁ KẾT QUẢ

**1. Đánh giá về quá trình Feature Selection (Chọn đặc trưng)**
- **Giảm chiều dữ liệu hiệu quả:** Việc áp dụng `SelectFromModel` với thuật toán Random Forest đã giúp loại bỏ đáng kể các biến mang ít thông tin dự báo hoặc bị nhiễu (noise) sinh ra sau quá trình One-Hot Encoding. 
- **Bảo toàn thông tin cốt lõi:** Các đặc trưng giữ lại chủ yếu là những biến có độ tương quan mạnh với giá nhà (SalePrice) đã được phân tích từ Lab 03 như `TotalSF` (Tổng diện tích), `OverallQual` (Chất lượng tổng thể), và `HouseAge` (Tuổi thọ nhà). Điều này giúp mô hình tập trung học từ các đặc trưng có giá trị thực sự, tránh hiện tượng học vẹt (Overfitting).

**2. So sánh hiệu suất các mô hình Machine Learning**
Dựa vào chỉ số RMSE (Root Mean Squared Error - Càng thấp càng tốt) và R2 Score (Hệ số xác định - Càng gần 1.0 càng tốt) trên tập kiểm thử, ta có các nhận xét sau:

*   **XGBoost (XGBRegressor):** 
    * Thường cho hiệu suất **tốt nhất** trong 3 mô hình. 
    * *Lý do:* Thuật toán Gradient Boosting xây dựng các cây quyết định tuần tự, cây sau sửa lỗi cho cây trước. Kết hợp với `learning_rate` nhỏ (0.05), mô hình hội tụ tốt, nắm bắt xuất sắc các mối quan hệ phức tạp và phi tuyến tính của dữ liệu giá nhà.
*   **Random Forest:** 
    * Cho kết quả **rất ổn định** và là một baseline (mô hình cơ sở) vững chắc. 
    * *Lý do:* Thuật toán Bagging giúp giảm phương sai (variance) bằng cách lấy trung bình dự đoán của nhiều cây quyết định độc lập. Tuy nhiên, nó không tối ưu hóa trực tiếp vào các sai số (residuals) như XGBoost nên RMSE thường nhỉnh hơn một chút.
*   **Support Vector Machine (SVR):** 
    * Hiệu suất thường **thấp hơn** so với hai mô hình Tree-based (XGBoost, Random Forest) trên tập dữ liệu này. 
    * *Lý do:* Dù đã áp dụng `StandardScaler` để chuẩn hóa (vì SVM rất nhạy cảm với thang đo), SVM vẫn gặp khó khăn nhất định với các bộ dữ liệu dạng bảng (tabular data) có chứa nhiều biến phân loại (categorical) đã qua One-Hot Encoding. Thuật toán này tỏa sáng hơn ở các không gian đặc trưng liên tục, mật độ cao thay vì không gian thưa thớt (sparse matrix) do biến giả tạo ra.

**=> Kết luận:** Đối với bài toán hồi quy dự đoán giá nhà (Advanced Regression Techniques) với nhiều đặc trưng hỗn hợp, **XGBoost** là lựa chọn tối ưu nhất trong bộ 3 thuật toán thử nghiệm. Quá trình chọn lọc đặc trưng (Feature Selection) đóng vai trò bản lề giúp mô hình tăng tốc độ huấn luyện và cải thiện độ khái quát hóa.